In [ ]:
"""
Monthly Sales Forecasting System
=================================
Predicts month-end results based on daily actuals + historical patterns.

4 Layers:
  1. Weighted Daily Run Rate
  2. Day-of-Week Seasonality Index
  3. Multi-Metric Tracking (Revenue, Orders, Ad Spend, CRM)
  4. Pacing Score & Action Signals

Usage:
  1. Fill MONTHLY_TARGET with your targets
  2. Fill HISTORICAL_DATA with past months' daily averages by day-of-week
  3. Update DAILY_ACTUALS as each day passes
  4. Run the script → get forecast + pacing + action signals
"""

import datetime
import calendar
from dataclasses import dataclass


# ╔══════════════════════════════════════════════════════════════════╗
# ║                     CONFIGURATION                               ║
# ╚══════════════════════════════════════════════════════════════════╝

# Current month info
YEAR = 2026
MONTH = 3

# Monthly targets per metric
MONTHLY_TARGET = {
    "revenue": 500_000,
    "orders": 2_000,
    "ad_spend": 80_000,
    "crm_clicks": 15_000,
}

# Historical daily averages BY DAY OF WEEK (from past 3-6 months)
# Keys: 0=Monday, 1=Tuesday, ..., 5=Saturday, 6=Sunday
HISTORICAL_DAILY_AVG = {
    "revenue": {
        0: 15_200, 1: 14_800, 2: 16_500, 3: 17_100,
        4: 19_800, 5: 21_500, 6: 18_000,
    },
    "orders": {
        0: 62, 1: 58, 2: 67, 3: 70,
        4: 80, 5: 88, 6: 73,
    },
    "ad_spend": {
        0: 2_600, 1: 2_500, 2: 2_700, 3: 2_800,
        4: 3_000, 5: 3_200, 6: 2_900,
    },
    "crm_clicks": {
        0: 480, 1: 450, 2: 510, 3: 530,
        4: 600, 5: 650, 6: 550,
    },
}

# Daily actuals — add a new row each day
# Format: (day_of_month, revenue, orders, ad_spend, crm_clicks)
DAILY_ACTUALS = [
    (1,    18_200,   75,     2_800,    520),
    (2,    16_500,   68,     2_600,    490),
    (3,    15_800,   63,     2_500,    470),
    (4,    17_900,   72,     2_750,    530),
    (5,    20_100,   82,     3_100,    610),
    (6,    22_300,   90,     3_300,    680),
    (7,    19_000,   77,     2_900,    560),
    (8,    18_800,   76,     2_850,    540),
    (9,    17_200,   69,     2_650,    500),
    (10,   16_900,   67,     2_600,    485),
    (11,   19_500,   79,     2_950,    570),
    (12,   21_800,   88,     3_200,    650),
]


# ╔══════════════════════════════════════════════════════════════════╗
# ║                     CORE ENGINE                                  ║
# ╚══════════════════════════════════════════════════════════════════╝

METRIC_KEYS = ["revenue", "orders", "ad_spend", "crm_clicks"]
RECENT_WINDOW = 4


@dataclass
class MetricForecast:
    name: str
    actual_so_far: float
    forecasted_total: float
    target: float
    pacing_score: float
    trend: str
    status: str
    action: str
    confidence: float


def build_seasonality_index(historical: dict[int, float]) -> dict[int, float]:
    """Layer 2: Historical day-of-week averages → seasonality index (1.0 = average)."""
    overall_avg = sum(historical.values()) / len(historical)
    if overall_avg == 0:
        return {d: 1.0 for d in range(7)}
    return {day: val / overall_avg for day, val in historical.items()}


def weighted_daily_rate(daily_values: list[float], recent_window: int = RECENT_WINDOW) -> float:
    """Layer 1: 70% weight on recent days, 30% on earlier days."""
    if not daily_values:
        return 0.0
    if len(daily_values) <= recent_window:
        return sum(daily_values) / len(daily_values)

    recent = daily_values[-recent_window:]
    earlier = daily_values[:-recent_window]
    return (sum(recent) / len(recent)) * 0.70 + (sum(earlier) / len(earlier)) * 0.30


def get_remaining_weekdays(year: int, month: int, last_completed_day: int) -> list[int]:
    """List of weekday indices (0=Mon..6=Sun) for remaining days in the month."""
    total_days = calendar.monthrange(year, month)[1]
    return [
        datetime.date(year, month, d).weekday()
        for d in range(last_completed_day + 1, total_days + 1)
    ]


def detect_trend(daily_values: list[float], window: int = 4) -> str:
    """Compare recent vs earlier window to detect direction."""
    if len(daily_values) < window * 2:
        if len(daily_values) >= 4:
            mid = len(daily_values) // 2
            avg_first = sum(daily_values[:mid]) / mid
            avg_second = sum(daily_values[mid:]) / (len(daily_values) - mid)
            change = (avg_second - avg_first) / avg_first if avg_first else 0
        else:
            return "→ flat (not enough data)"
    else:
        prev = daily_values[-(window * 2):-window]
        recent = daily_values[-window:]
        avg_prev = sum(prev) / len(prev)
        avg_recent = sum(recent) / len(recent)
        change = (avg_recent - avg_prev) / avg_prev if avg_prev else 0

    if change > 0.05:
        return f"↑ improving (+{change:.0%})"
    elif change < -0.05:
        return f"↓ declining ({change:.0%})"
    else:
        return f"→ flat ({change:+.0%})"


def compute_confidence(day_number: int, total_days: int) -> float:
    """Confidence curve: ramps up as more data is available."""
    p = day_number / total_days
    if p <= 0.25:
        return 30 + (p / 0.25) * 20
    elif p <= 0.50:
        return 50 + ((p - 0.25) / 0.25) * 25
    elif p <= 0.85:
        return 75 + ((p - 0.50) / 0.35) * 15
    else:
        return 90 + ((p - 0.85) / 0.15) * 10


def pacing_status(score: float, metric_name: str) -> tuple[str, str]:
    """Layer 4: Status emoji + action signal. Ad spend is inverted (lower = better)."""
    if metric_name == "ad_spend":
        if score < 95:
            return "🟢 Under Budget", "continue — efficient spending"
        elif score <= 105:
            return "🟡 On Budget", "continue — monitor closely"
        elif score <= 115:
            return "🟠 Over Budget", "scale back — optimize targeting"
        else:
            return "🔴 Over Budget", "reduce spend — check ROAS immediately"
    else:
        if score > 105:
            return "🟢 Ahead", "continue — maintain momentum"
        elif score >= 95:
            return "🟡 On Track", "continue — stay consistent"
        elif score >= 85:
            return "🟠 Behind", "push harder — increase effort"
        else:
            return "🔴 At Risk", "urgent action — re-evaluate strategy"


def forecast_metric(
    metric_name: str,
    daily_values: list[float],
    target: float,
    historical: dict[int, float],
    year: int,
    month: int,
    last_day: int,
) -> MetricForecast:
    """Full 4-layer forecast for one metric."""
    total_days = calendar.monthrange(year, month)[1]
    actual_so_far = sum(daily_values)

    # Layer 1 + 2: Weighted rate × seasonality for remaining days
    w_rate = weighted_daily_rate(daily_values)
    season_index = build_seasonality_index(historical)
    remaining_weekdays = get_remaining_weekdays(year, month, last_day)

    remaining_forecast = sum(
        w_rate * season_index.get(wd, 1.0) for wd in remaining_weekdays
    )
    forecasted_total = actual_so_far + remaining_forecast

    # Layer 4: Pacing — seasonality-adjusted expected value by today
    days_so_far_wd = [
        datetime.date(year, month, d).weekday() for d in range(1, last_day + 1)
    ]
    all_month_wd = [
        datetime.date(year, month, d).weekday() for d in range(1, total_days + 1)
    ]
    full_month_weight = sum(season_index.get(wd, 1.0) for wd in all_month_wd)
    so_far_weight = sum(season_index.get(wd, 1.0) for wd in days_so_far_wd)
    expected_by_today = target * (so_far_weight / full_month_weight)

    pacing_score = (actual_so_far / expected_by_today * 100) if expected_by_today else 100
    status, action = pacing_status(pacing_score, metric_name)

    return MetricForecast(
        name=metric_name,
        actual_so_far=actual_so_far,
        forecasted_total=forecasted_total,
        target=target,
        pacing_score=pacing_score,
        trend=detect_trend(daily_values),
        status=status,
        action=action,
        confidence=compute_confidence(last_day, total_days),
    )


# ╔══════════════════════════════════════════════════════════════════╗
# ║                     DERIVED KPIs (Layer 3)                       ║
# ╚══════════════════════════════════════════════════════════════════╝

def compute_derived_kpis(forecasts: dict[str, MetricForecast]) -> dict:
    rev = forecasts["revenue"].forecasted_total
    orders = forecasts["orders"].forecasted_total
    spend = forecasts["ad_spend"].forecasted_total
    clicks = forecasts["crm_clicks"].forecasted_total

    return {
        "AOV (Avg Order Value)": round(rev / orders, 2) if orders else 0,
        "ROAS (Return on Ad Spend)": round(rev / spend, 2) if spend else 0,
        "CPA (Cost per Acquisition)": round(spend / orders, 2) if orders else 0,
        "Conv% (Orders / CRM Clicks)": round((orders / clicks) * 100, 2) if clicks else 0,
    }


# ╔══════════════════════════════════════════════════════════════════╗
# ║                     REPORT                                       ║
# ╚══════════════════════════════════════════════════════════════════╝

def run_forecast():
    metric_daily = {k: [] for k in METRIC_KEYS}
    for row in DAILY_ACTUALS:
        day, rev, orders, spend, clicks = row
        metric_daily["revenue"].append(rev)
        metric_daily["orders"].append(orders)
        metric_daily["ad_spend"].append(spend)
        metric_daily["crm_clicks"].append(clicks)

    last_day = DAILY_ACTUALS[-1][0]
    total_days = calendar.monthrange(YEAR, MONTH)[1]
    month_name = calendar.month_name[MONTH]

    print("=" * 70)
    print(f"  📊 MONTHLY FORECAST REPORT — {month_name} {YEAR}")
    print(f"  Day {last_day} of {total_days} ({last_day/total_days:.0%} of month complete)")
    print("=" * 70)

    forecasts = {}
    for metric in METRIC_KEYS:
        forecasts[metric] = forecast_metric(
            metric_name=metric,
            daily_values=metric_daily[metric],
            target=MONTHLY_TARGET[metric],
            historical=HISTORICAL_DAILY_AVG[metric],
            year=YEAR,
            month=MONTH,
            last_day=last_day,
        )

    for metric in METRIC_KEYS:
        f = forecasts[metric]
        vs_target = ((f.forecasted_total - f.target) / f.target) * 100
        label = f.name.upper().replace("_", " ")

        print(f"\n  ┌─── {label} {'─' * (50 - len(label))}┐")
        print(f"  │  Actual so far:     {f.actual_so_far:>12,.0f}")
        print(f"  │  Forecasted total:  {f.forecasted_total:>12,.0f}")
        print(f"  │  Target:            {f.target:>12,.0f}")
        print(f"  │  vs Target:         {vs_target:>+11.1f}%")
        print(f"  │  Pacing:            {f.pacing_score:>11.1f}%  {f.status}")
        print(f"  │  Trend:             {f.trend}")
        print(f"  │  Confidence:        {f.confidence:>11.0f}%")
        print(f"  │  ➤ Action:          {f.action}")
        print(f"  └{'─' * 58}┘")

    # Derived KPIs
    kpis = compute_derived_kpis(forecasts)
    print(f"\n  {'─' * 58}")
    print("  📈 FORECASTED KPIs (Month-End)")
    print(f"  {'─' * 58}")
    for name, value in kpis.items():
        if "%" in name:
            print(f"    {name:<35} {value:>8.2f}%")
        elif "ROAS" in name:
            print(f"    {name:<35} {value:>8.2f}x")
        else:
            print(f"    {name:<35} {value:>8,.2f}")

    # Overall health
    print(f"\n  {'─' * 58}")
    print("  🎯 OVERALL HEALTH")
    print(f"  {'─' * 58}")
    rev_f = forecasts["revenue"]
    spend_f = forecasts["ad_spend"]

    if rev_f.pacing_score >= 95 and spend_f.pacing_score <= 105:
        print("    ✅ Healthy — Revenue on track, spend under control")
    elif rev_f.pacing_score >= 95 and spend_f.pacing_score > 105:
        print("    ⚠️  Revenue OK but overspending — check ROAS & optimize")
    elif rev_f.pacing_score < 95 and spend_f.pacing_score <= 105:
        print("    ⚠️  Revenue behind but spend OK — push CRM & organic")
    else:
        print("    🚨 Revenue behind AND overspending — pause & reassess")

    print(f"\n{'=' * 70}")
    print(f"  Confidence: {rev_f.confidence:.0f}% | Next: Enter Day {last_day + 1} and re-run")
    print(f"{'=' * 70}\n")


if __name__ == "__main__":
    run_forecast()